# CareerPilot Placement Matching Engine: Evaluation & Ranking Benchmark

This notebook demonstrates the quantitative evaluation of the **CareerPilot Placement Matching Engine** against a curated, frozen benchmark of entry-level campus placement roles and candidate profiles.

### Core Evaluation Metrics:
- **Mean Reciprocal Rank (MRR)**: Measures how early the first relevant job appears in the ranked candidate recommendations ($1/\text{rank}$). Target: $\ge 0.90$.
- **Recall@K (K=1, 3, 5)**: Proportion of all relevant jobs retrieved in the top $K$ recommendations. Target Recall@3: $\ge 0.85$.
- **Normalized Discounted Cumulative Gain (nDCG@K)**: Measures graded ranking quality, heavily penalizing high-relevance jobs ranked too low. Target nDCG@3: $\ge 0.90$.
- **Precision@K (K=1, 3)**: Proportion of retrieved items in the top $K$ that are relevant. Target Precision@1: $\ge 0.80$.

> **Ethical AI Disclaimer**: CareerPilot match scores evaluate lexical and semantic alignment between candidate resumes and job skill requirements for educational and diagnostic purposes. They do not predict hiring decisions or guarantee job placement.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd

# Ensure project root and backend modules are in python path
REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT / "backend") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "backend"))
if str(REPO_ROOT / "evaluation") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "evaluation"))

from app.services.matching_service import calculate_lexical_match
from run_matching_eval import (
    compute_dcg_at_k,
    compute_ndcg_at_k,
    compute_recall_at_k,
    compute_mrr,
    compute_precision_at_k,
    run_matching_evaluation,
    print_evaluation_summary,
)

print("Matching service and evaluation harness successfully imported.")

## 1. Load Frozen Benchmark Dataset

The benchmark dataset (`evaluation/data/matching_benchmark.json`) consists of:
- **5 Diverse Job Roles**: Junior Python Backend, Entry-Level Data Analyst, Junior Frontend Developer, Cloud & DevOps Apprentice, Junior Machine Learning Engineer.
- **5 Representative Candidates**: With explicit resumes, extracted canonical skills, and ground-truth relevance annotations ($0 = \text{irrelevant}$, $1 = \text{marginal}$, $2 = \text{moderately relevant}$, $3 = \text{highly relevant}$).

In [ ]:
benchmark_file = REPO_ROOT / "evaluation" / "data" / "matching_benchmark.json"
with open(benchmark_file, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

print(f"Loaded benchmark version: {benchmark['version']}")
print(f"Total Jobs: {len(benchmark['jobs'])}")
print(f"Total Candidates: {len(benchmark['candidates'])}")

# Display jobs summary table
jobs_df = pd.DataFrame([
    {
        "ID": j["id"],
        "Title": j["title"],
        "Company": j["company"],
        "Required Skills": ", ".join(j["required_skills"]),
        "Preferred Skills": ", ".join(j["preferred_skills"]),
    }
    for j in benchmark["jobs"]
])
jobs_df

## 2. Inspect Ground Truth Relevance Labels

Here we view candidate target roles, skills, and their ground truth relevance scores against all benchmark job openings.

In [ ]:
gt_rows = []
for c in benchmark["candidates"]:
    row = {
        "Candidate": c["name"],
        "Target Role": c["target_role"],
        "Candidate Skills": ", ".join(c["skills"]),
    }
    for job_id, rel in c["ground_truth_relevance"].items():
        row[job_id] = rel
    gt_rows.append(row)

gt_df = pd.DataFrame(gt_rows)
gt_df

## 3. Execute End-to-End Evaluation

We run the evaluation pipeline across all candidates and job openings. The pipeline:
1. Computes composite lexical match scores combining required skill coverage (60%), preferred coverage (20%), and TF-IDF cosine similarity (20%).
2. Ranks all jobs for each candidate.
3. Calculates MRR, Recall@1/3/5, Precision@1/3, and nDCG@3/5 against ground truth relevance.

In [ ]:
report = run_matching_evaluation(
    benchmark_path=str(benchmark_file),
    output_path=str(REPO_ROOT / "evaluation" / "results" / "matching_metrics.json"),
    use_semantic=False,
)

print_evaluation_summary(report)

## 4. Aggregate Macro Performance Metrics

The table below summarizes system performance across all benchmark queries.

In [ ]:
macro_df = pd.DataFrame([
    {"Metric": "Mean Reciprocal Rank (MRR)", "Score": report["macro_metrics"]["mean_mrr"], "Interpretation": "First relevant job rank quality"},
    {"Metric": "Mean Recall@1", "Score": report["macro_metrics"]["mean_recall_at_1"], "Interpretation": "Relevant jobs retrieved at rank 1"},
    {"Metric": "Mean Recall@3", "Score": report["macro_metrics"]["mean_recall_at_3"], "Interpretation": "Relevant jobs retrieved in top 3"},
    {"Metric": "Mean Recall@5", "Score": report["macro_metrics"]["mean_recall_at_5"], "Interpretation": "Relevant jobs retrieved in top 5"},
    {"Metric": "Mean Precision@1", "Score": report["macro_metrics"]["mean_precision_at_1"], "Interpretation": "Top-1 hit accuracy"},
    {"Metric": "Mean Precision@3", "Score": report["macro_metrics"]["mean_precision_at_3"], "Interpretation": "Density of relevant jobs in top 3"},
    {"Metric": "Mean nDCG@3", "Score": report["macro_metrics"]["mean_ndcg_at_3"], "Interpretation": "Graded ranking quality at top 3"},
    {"Metric": "Mean nDCG@5", "Score": report["macro_metrics"]["mean_ndcg_at_5"], "Interpretation": "Graded ranking quality across full set"},
])
macro_df

## 5. Explainability & Skill Gap Deep-Dive

Inspect the top recommendation for Alex Chen (`cand_python_star`) to confirm explainability and transparent score breakdown.

In [ ]:
alex_res = report["candidate_results"][0]
print(f"Candidate: {alex_res['candidate_name']} ({alex_res['target_role']})")
print("\nRanked Job Matches:")
for i, job in enumerate(alex_res["ranked_jobs"], start=1):
    print(f"{i}. {job['job_title']} (ID: {job['job_id']})")
    print(f"   Overall Score: {job['overall_score']:.1f}% | Ground Truth: {job['ground_truth_relevance']}")
    print(f"   Req Cov: {job['required_coverage']:.0%} | Pref Cov: {job['preferred_coverage']:.0%} | TF-IDF Sim: {job['tfidf_similarity']:.4f}")
    print(f"   Matched Req: {job['matched_required']}")
    print(f"   Missing Req: {job['missing_required']}")
    print()